# ESCI reranker inference

Load trained reranker and score (query, product) pairs for a sample query from the test set.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.constants import DATA_DIR
from src.data.load_data import prepare_train_test
from src.models.reranker import load_reranker

In [ ]:
# Load reranker and test data
import pandas as pd

reranker = load_reranker(model_path=DATA_DIR / "reranker")
test_path = DATA_DIR / "esci_test.parquet"
if test_path.exists():
    test_df = pd.read_parquet(test_path)
else:
    _, test_df = prepare_train_test(data_dir=DATA_DIR)

# Pick a sample query
sample = test_df.groupby("query_id").first().reset_index().iloc[0]
query = str(sample["query"])
qid = sample["query_id"]
print(f"Query: {query}")

In [ ]:
# Get all products for this query and rerank
rows = test_df[test_df["query_id"] == qid]
candidates = [(str(r["product_id"]), str(r["product_text"])) for _, r in rows.iterrows()]
ranked = reranker.rerank(query, candidates, batch_size=32)

# Show top 5 with labels
pid_to_label = dict(zip(rows["product_id"].astype(str), rows["esci_label"]))
for rank, (pid, score) in enumerate(ranked[:5], start=1):
    label = pid_to_label.get(pid, "?")
    text = next(t for p, t in candidates if p == pid)
    print(f"{rank}. [{label}] score={score:.4f}")
    print(f"    {text[:200]}...")
    print()